In [ ]:
# Cell 1 : Import Libraries

import os
import ta
import json
import joblib
import yfinance as yf
import time
import logging
from tqdm import tqdm
import pandas as pd
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from datetime import timedelta
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from sklearn.model_selection import train_test_split
from scipy.stats import skew, kurtosis, shapiro
from sklearn.metrics import explained_variance_score, r2_score
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt

In [ ]:
# Cell 2: Fetch the Stock Data

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Initialize dictionaries to store dataframes
daily_data_dict = {}
hourly_data_dict = {}
additional_info_dict = {}
financials_dict = {}
balance_sheet_dict = {}
cash_flow_dict = {}

# List of stocks to fetch data for
stocks = [
    'AAPL', 'MSFT', 'GOOGL', 'NVDA', 'AMZN', 'META', 'JNJ', 'PFE', 'UNH',
    'JPM', 'BAC', 'GS', 'PG', 'KO', 'WMT', 'XOM', 'CVX', 'BA', 'HON', 'VZ'
]

# Define the time frames for data
end_date = datetime.now()
start_date_daily = end_date - timedelta(days=4*365)
start_date_hourly = end_date - timedelta(days=729)

# Create directories for the data
os.makedirs('../data/missing_data', exist_ok=True) 
os.makedirs('../data/stock_data', exist_ok=True)

# Function to fetch stock data
def fetch_stock_data(ticker, start, end, interval):
    try:
        data = yf.download(ticker, start=start, end=end, interval=interval)
        if data.empty:
            logging.warning(f"No data retrieved for {ticker} from {start} to {end} with interval {interval}")
        return data.drop(columns=['Adj Close'], errors='ignore')
    except Exception as e:
        logging.error(f"Error fetching data for {ticker}: {e}")
        return pd.DataFrame()

# Function to fetch additional fundamental data
def fetch_additional_data(ticker):
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        additional_info = {
            'PE_Ratio': info.get('trailingPE', np.nan),
            'Market_Cap': info.get('marketCap', np.nan),
            'Dividend_Yield': info.get('dividendYield', np.nan),
            'EPS': info.get('trailingEps', np.nan),
            'Beta': info.get('beta', np.nan),
            'PB_Ratio': info.get('priceToBook', np.nan)
        }

        # Log missing data for critical fields only
        missing_data = [k for k, v in additional_info.items() if pd.isna(v)]
        if missing_data:
            logging.warning(f"Missing data for {ticker}: {missing_data}")

        financials = stock.financials.T
        balance_sheet = stock.balance_sheet.T
        cash_flow = stock.cashflow.T

        return additional_info, financials, balance_sheet, cash_flow
    except Exception as e:
        logging.error(f"Error fetching additional data for {ticker}: {e}")
        return {}, pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

# Fetch and save data
for idx, stock in enumerate(tqdm(stocks, desc="Fetching stocks data")):
    
    # Daily Data (3 years)
    daily_data = fetch_stock_data(stock, start_date_daily, end_date, '1d')
    daily_data_dict[stock] = daily_data
    daily_data.to_csv(f'../data/stock_data/{stock}_daily.csv', index=True)

    # Hourly Data (2 years, actually 729 days)
    hourly_data = fetch_stock_data(stock, start_date_hourly, end_date, '1h')
    hourly_data_dict[stock] = hourly_data
    hourly_data.to_csv(f'../data/stock_data/{stock}_hourly.csv', index=True)

    # Additional Fundamental Data (3 years)
    additional_info, financials, balance_sheet, cash_flow = fetch_additional_data(stock)
    additional_info_dict[stock] = additional_info
    financials_dict[stock] = financials
    balance_sheet_dict[stock] = balance_sheet
    cash_flow_dict[stock] = cash_flow

    # Save the financial data to CSV files
    financials.to_csv(f'../data/stock_data/{stock}_financials.csv', index=True)
    balance_sheet.to_csv(f'../data/stock_data/{stock}_balance_sheet.csv', index=True)
    cash_flow.to_csv(f'../data/stock_data/{stock}_cash_flow.csv', index=True)
    
    # Convert additional_info to DataFrame and save it
    additional_info_df = pd.DataFrame([additional_info])
    additional_info_df.to_csv(f'../data/stock_data/{stock}_additional_info.csv', index=False)

    time.sleep(2)

print("Data fetching and saving complete.")

In [ ]:
# Cell 3: Drop Irrelevant Columns, 2019 Data, and Specific Columns for All Stocks

# List of columns to drop based on previous analysis
columns_to_drop = [
    'Treasury Shares Number', 'Deferred Tax Assets', 'Deferred Tax Liabilities', 
    'Prepaid Assets', 'Employee Benefits', 'Inventory Adjustments', 'Other Receivables', 
    'Gain/Loss on Sale of Assets', 'Current Deferred Liabilities', 'Deferred Revenue', 
    'Non-Current Prepaid Assets', 'Other Intangible Assets', 'Ordinary Shares Number', 
    'Other Non Current Liabilities', 'Gross PPE', 'Net PPE', 'Net Tangible Assets', 
    'Other Equity Adjustments', 'Other Short Term Investments', 'Total Tax Payable', 
    'Other Investments', 'Net PPE Purchase And Sale', 'Repurchase Of Capital Stock', 
    'Gain Loss On Investment Securities', 'Other Non Cash Items',
    # Additional columns to drop
    'Special Income Charges', 'Other Special Charges', 'Other Payable', 
    'Inventory', 'Taxes Receivable', 'Net Intangibles Purchase and Sale', 
    'Purchase of Intangibles', 'Amortization Cash Flow', 
    'Amortization of Intangibles'
]

# Function to drop columns, remove 2019 data, and show stats before and after
def drop_columns_and_remove_2019(df_dict, columns_to_drop):
    stats_before = {}
    stats_after = {}
    
    for stock, df in df_dict.items():
        if not df.empty:  # Ensure there is data to process
            # Capture stats before dropping columns and 2019 data
            stats_before[stock] = {
                'Total Columns': df.shape[1],
                'Missing Values': df.isnull().sum().sum()
            }
            # Drop the irrelevant columns
            df_dropped = df.drop(columns=columns_to_drop, errors='ignore')
            
            # Drop the 2019 data by filtering out that year from the index
            df_dropped = df_dropped[df_dropped.index >= '2021-01-01']
            
            # Capture stats after dropping columns and 2019 data
            stats_after[stock] = {
                'Total Columns': df_dropped.shape[1],
                'Missing Values': df_dropped.isnull().sum().sum()
            }
            
            # Update the dataframe in the dictionary
            df_dict[stock] = df_dropped
    
    return stats_before, stats_after

# Apply the function to all financial dataframes
financials_before, financials_after = drop_columns_and_remove_2019(financials_dict, columns_to_drop)
balance_sheet_before, balance_sheet_after = drop_columns_and_remove_2019(balance_sheet_dict, columns_to_drop)
cash_flow_before, cash_flow_after = drop_columns_and_remove_2019(cash_flow_dict, columns_to_drop)

In [ ]:
# Cell 4 : Outliers Detection

# Initialize dictionaries to store the processed data
daily_outliers_dict = {}
daily_clean_dict = {}
hourly_outliers_dict = {}
hourly_clean_dict = {}

# Updated function to detect outliers using IQR method
def detect_outliers_and_store(df):
    # Drop any missing values to avoid issues during quantile calculation
    df = df.dropna()

    # Calculate IQR for each column
    Q1 = df.quantile(0.25)
    Q3 = df.quantile(0.75)
    IQR = Q3 - Q1

    # Determine outliers across all columns
    outliers = ((df < (Q1 - 1.5 * IQR)) | (df > (Q3 + 1.5 * IQR))).any(axis=1)
    
    return df[outliers], df[~outliers]

# Apply to time series data
for stock in stocks:
    # Load the daily and hourly data from the processed data
    daily_data = daily_data_dict.get(stock)
    hourly_data = hourly_data_dict.get(stock)
    
    if daily_data is not None:
        # Detect outliers in daily data
        daily_outliers, daily_clean = detect_outliers_and_store(daily_data)
        daily_outliers_dict[stock] = daily_outliers
        daily_clean_dict[stock] = daily_clean
    
    if hourly_data is not None:
        # Detect outliers in hourly data
        hourly_outliers, hourly_clean = detect_outliers_and_store(hourly_data)
        hourly_outliers_dict[stock] = hourly_outliers
        hourly_clean_dict[stock] = hourly_clean

    logging.info(f"Processed outliers for stock: {stock}")

print("Outlier detection complete using IQR method. DataFrames are ready for analysis.")


In [ ]:
# Cell 5(a): Inspect Outlier Statistics for Both Daily and Hourly Data

def inspect_outlier_stats(daily_outliers_dict, daily_clean_dict, hourly_outliers_dict, hourly_clean_dict):
    total_outliers_daily = 0
    total_data_daily = 0
    total_outliers_hourly = 0
    total_data_hourly = 0

    # Inspecting Daily Data Outliers
    print("Inspecting Daily Data Outliers:")
    for stock in daily_outliers_dict.keys():
        print(f"\nStatistics for {stock} (Daily Data):")
        
        if not daily_outliers_dict[stock].empty:
            # Calculate statistics for daily outliers
            outlier_stats_daily = daily_outliers_dict[stock].describe().T
            print("Daily Outlier Statistics:")
            display(outlier_stats_daily)
        else:
            print("No outliers found in Daily Data for this stock.")
        
        if not daily_clean_dict[stock].empty:
            # Calculate statistics for daily non-outliers
            non_outlier_stats_daily = daily_clean_dict[stock].describe().T
            print("Daily Non-Outlier Statistics:")
            display(non_outlier_stats_daily)
        else:
            print("No non-outliers found in Daily Data for this stock.")
        
        # Summary of outlier percentage
        total_daily = len(daily_data_dict[stock])
        outliers_daily = len(daily_outliers_dict[stock])
        total_outliers_daily += outliers_daily
        total_data_daily += total_daily
        print(f"Outliers Percentage (Daily Data): {outliers_daily / total_daily * 100:.2f}%")
        
        print("-" * 80)
    
    # Inspecting Hourly Data Outliers
    print("\nInspecting Hourly Data Outliers:")
    for stock in hourly_outliers_dict.keys():
        print(f"\nStatistics for {stock} (Hourly Data):")
        
        if not hourly_outliers_dict[stock].empty:
            # Calculate statistics for hourly outliers
            outlier_stats_hourly = hourly_outliers_dict[stock].describe().T
            print("Hourly Outlier Statistics:")
            display(outlier_stats_hourly)
        else:
            print("No outliers found in Hourly Data for this stock.")
        
        if not hourly_clean_dict[stock].empty:
            # Calculate statistics for hourly non-outliers
            non_outlier_stats_hourly = hourly_clean_dict[stock].describe().T
            print("Hourly Non-Outlier Statistics:")
            display(non_outlier_stats_hourly)
        else:
            print("No non-outliers found in Hourly Data for this stock.")
        
        # Summary of outlier percentage
        total_hourly = len(hourly_data_dict[stock])
        outliers_hourly = len(hourly_outliers_dict[stock])
        total_outliers_hourly += outliers_hourly
        total_data_hourly += total_hourly
        print(f"Outliers Percentage (Hourly Data): {outliers_hourly / total_hourly * 100:.2f}%")
        
        print("-" * 80)
    
    # Overall Summary
    print("\nOverall Summary:")
    print(f"Total Daily Data Points: {total_data_daily}")
    print(f"Total Daily Outliers: {total_outliers_daily} ({total_outliers_daily / total_data_daily * 100:.2f}%)")
    print(f"Total Hourly Data Points: {total_data_hourly}")
    print(f"Total Hourly Outliers: {total_outliers_hourly} ({total_outliers_hourly / total_data_hourly * 100:.2f}%)")

# Run the inspection function for both daily and hourly data
inspect_outlier_stats(daily_outliers_dict, daily_clean_dict, hourly_outliers_dict, hourly_clean_dict)

In [ ]:
# Cell 5(b): Plot Outliers for Visual Inspection for Both Daily and Hourly Data

# Function to plot outliers for each stock in both daily and hourly data
def plot_outliers(daily_outliers_dict, daily_clean_dict, hourly_outliers_dict, hourly_clean_dict):
    for stock in daily_outliers_dict.keys():
        # Plot Daily Data
        print(f"Plotting outliers for {stock} (Daily Data):")
        daily_outliers = daily_outliers_dict[stock]
        daily_clean = daily_clean_dict[stock]

        plt.figure(figsize=(14, 7))
        plt.plot(daily_clean.index, daily_clean['Close'], label='Clean Data (Daily)', color='blue')
        plt.scatter(daily_outliers.index, daily_outliers['Close'], label='Outliers (Daily)', color='red', marker='x')
        plt.title(f'Daily Outliers in Stock Data for {stock}')
        plt.xlabel('Date')
        plt.ylabel('Closing Price')
        plt.legend()
        plt.grid(True)
        plt.show()

        # Plot Hourly Data
        if stock in hourly_outliers_dict:
            print(f"Plotting outliers for {stock} (Hourly Data):")
            hourly_outliers = hourly_outliers_dict[stock]
            hourly_clean = hourly_clean_dict[stock]

            plt.figure(figsize=(14, 7))
            plt.plot(hourly_clean.index, hourly_clean['Close'], label='Clean Data (Hourly)', color='green')
            plt.scatter(hourly_outliers.index, hourly_outliers['Close'], label='Outliers (Hourly)', color='orange', marker='x')
            plt.title(f'Hourly Outliers in Stock Data for {stock}')
            plt.xlabel('Datetime')
            plt.ylabel('Closing Price')
            plt.legend()
            plt.grid(True)
            plt.show()

# Run the plot function for both daily and hourly data
plot_outliers(daily_outliers_dict, daily_clean_dict, hourly_outliers_dict, hourly_clean_dict)

In [ ]:
# Cell 6 : Handling Outliers

# --- Daily Data Processing ---

def apply_robust_scaling_daily(daily_df):
    """
    Apply RobustScaler to daily data to scale features while minimizing the impact of outliers.
    """
    scaler = RobustScaler()
    daily_scaled_df = pd.DataFrame(scaler.fit_transform(daily_df), columns=daily_df.columns, index=daily_df.index)
    return daily_scaled_df

def cap_outliers_daily(daily_df, lower_bound=1.5, upper_bound=1.5):
    """
    Cap outliers in daily data using IQR method.
    """
    Q1 = daily_df.quantile(0.25)
    Q3 = daily_df.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_cap = Q1 - lower_bound * IQR
    upper_cap = Q3 + upper_bound * IQR
    
    daily_capped_df = daily_df.clip(lower=lower_cap, upper=upper_cap, axis=1)
    
    return daily_capped_df

def add_outlier_flag_daily(daily_df, lower_bound=1.5, upper_bound=1.5):
    """
    Add a flag to daily data indicating whether each row contains outliers.
    """
    Q1 = daily_df.quantile(0.25)
    Q3 = daily_df.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound_values = Q1 - lower_bound * IQR
    upper_bound_values = Q3 + upper_bound * IQR
    
    daily_outlier_flagged = daily_df.copy()
    daily_outlier_flagged['OutlierFlag'] = ((daily_df < lower_bound_values) | (daily_df > upper_bound_values)).any(axis=1).astype(int)
    
    return daily_outlier_flagged

def remove_outliers_daily(daily_df, lower_bound=1.5, upper_bound=1.5):
    """
    Remove rows containing outliers in daily data.
    """
    Q1 = daily_df.quantile(0.25)
    Q3 = daily_df.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound_values = Q1 - lower_bound * IQR
    upper_bound_values = Q3 + upper_bound * IQR
    
    daily_no_outliers = daily_df[(daily_df >= lower_bound_values) & (daily_df <= upper_bound_values)].dropna()
    
    return daily_no_outliers

# --- Hourly Data Processing ---

def apply_robust_scaling_hourly(hourly_df):
    """
    Apply RobustScaler to hourly data to scale features while minimizing the impact of outliers.
    """
    scaler = RobustScaler()
    hourly_scaled_df = pd.DataFrame(scaler.fit_transform(hourly_df), columns=hourly_df.columns, index=hourly_df.index)
    return hourly_scaled_df

def cap_outliers_hourly(hourly_df, lower_bound=1.5, upper_bound=1.5):
    """
    Cap outliers in hourly data using IQR method.
    """
    Q1 = hourly_df.quantile(0.25)
    Q3 = hourly_df.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_cap = Q1 - lower_bound * IQR
    upper_cap = Q3 + upper_bound * IQR
    
    hourly_capped_df = hourly_df.clip(lower=lower_cap, upper=upper_cap, axis=1)
    
    return hourly_capped_df

def add_outlier_flag_hourly(hourly_df, lower_bound=1.5, upper_bound=1.5):
    """
    Add a flag to hourly data indicating whether each row contains outliers.
    """
    Q1 = hourly_df.quantile(0.25)
    Q3 = hourly_df.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound_values = Q1 - lower_bound * IQR
    upper_bound_values = Q3 + upper_bound * IQR
    
    hourly_outlier_flagged = hourly_df.copy()
    hourly_outlier_flagged['OutlierFlag'] = ((hourly_df < lower_bound_values) | (hourly_df > upper_bound_values)).any(axis=1).astype(int)
    
    return hourly_outlier_flagged

def remove_outliers_hourly(hourly_df, lower_bound=1.5, upper_bound=1.5):
    """
    Remove rows containing outliers in hourly data.
    """
    Q1 = hourly_df.quantile(0.25)
    Q3 = hourly_df.quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound_values = Q1 - lower_bound * IQR
    upper_bound_values = Q3 + upper_bound * IQR
    
    hourly_no_outliers = hourly_df[(hourly_df >= lower_bound_values) & (hourly_df <= upper_bound_values)].dropna()
    
    return hourly_no_outliers


In [ ]:
# Cell 7(a): Feature Engineering with Outliers (Separate for Daily and Hourly Data)

# Initialize dictionaries to store the feature-engineered data
daily_outlier_features_dict = {}
hourly_outlier_features_dict = {}
daily_ta_features_dict = {}
hourly_ta_features_dict = {}

def full_feature_engineering_with_outliers_daily(daily_df, lower_bound=1.5, upper_bound=1.5):
    if 'OutlierFlag' not in daily_df.columns:
        daily_df = add_outlier_flag_daily(daily_df, lower_bound=lower_bound, upper_bound=upper_bound)
    
    daily_df['RollingMean_10'] = daily_df['Close'].rolling(window=10).mean()
    daily_df['RollingStd_10'] = daily_df['Close'].rolling(window=10).std()
    daily_df.bfill(inplace=True)
    daily_df['LogReturn'] = np.log(daily_df['Close'] / daily_df['Close'].shift(1))
    daily_df['DaysSinceLastOutlier'] = (daily_df['OutlierFlag'] * np.arange(len(daily_df))).cummax() - np.arange(len(daily_df))
    daily_df['OutlierFreq_30'] = daily_df['OutlierFlag'].rolling(window=30).sum()
    
    return daily_df

def full_feature_engineering_with_outliers_hourly(hourly_df, lower_bound=1.5, upper_bound=1.5):
    if 'OutlierFlag' not in hourly_df.columns:
        hourly_df = add_outlier_flag_hourly(hourly_df, lower_bound=lower_bound, upper_bound=upper_bound)
    
    hourly_df['RollingMean_10'] = hourly_df['Close'].rolling(window=10).mean()
    hourly_df['RollingStd_10'] = hourly_df['Close'].rolling(window=10).std()
    hourly_df.bfill(inplace=True)
    hourly_df['LogReturn'] = np.log(hourly_df['Close'] / hourly_df['Close'].shift(1))
    hourly_df['DaysSinceLastOutlier'] = (hourly_df['OutlierFlag'] * np.arange(len(hourly_df))).cummax() - np.arange(len(hourly_df))
    hourly_df['OutlierFreq_30'] = hourly_df['OutlierFlag'].rolling(window=30).sum()
    
    return hourly_df

def apply_ta_features(df):
    df['SMA_20'] = ta.trend.sma_indicator(df['Close'], window=20)
    df['EMA_20'] = ta.trend.ema_indicator(df['Close'], window=20)
    df['RSI'] = ta.momentum.rsi(df['Close'], window=14)
    df['MACD'] = ta.trend.macd_diff(df['Close'])
    bb_indicator = ta.volatility.BollingerBands(df['Close'])
    df['BB_upper'] = bb_indicator.bollinger_hband()
    df['BB_middle'] = bb_indicator.bollinger_mavg()
    df['BB_lower'] = bb_indicator.bollinger_lband()
    df['ATR'] = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'], window=14)
    df['MFI'] = ta.volume.money_flow_index(df['High'], df['Low'], df['Close'], df['Volume'], window=14)
    
    df.bfill(inplace=True)
    
    return df

# Apply feature engineering to all stocks
for stock in stocks:
    # Feature engineering with outliers
    daily_outlier_features_dict[stock] = full_feature_engineering_with_outliers_daily(daily_data_dict[stock])
    hourly_outlier_features_dict[stock] = full_feature_engineering_with_outliers_hourly(hourly_data_dict[stock])
    
    # Apply technical analysis features
    daily_ta_features_dict[stock] = apply_ta_features(daily_outlier_features_dict[stock])
    hourly_ta_features_dict[stock] = apply_ta_features(hourly_outlier_features_dict[stock])

print("Feature engineering complete for all stocks.")

In [ ]:
# Cell 7(b): Feature Engineering of Time Series Data Using IQR-Based Outlier Handling and the TA Library

def apply_ta_features(df):
    """
    Apply technical analysis indicators to the stock data using the `ta` library.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing stock data with columns 'Open', 'High', 'Low', 'Close', 'Volume'.
    
    Returns:
    pd.DataFrame: DataFrame with new columns containing the calculated technical indicators.
    """
    df['SMA_20'] = ta.trend.sma_indicator(df['Close'], window=20)
    df['EMA_20'] = ta.trend.ema_indicator(df['Close'], window=20)
    df['RSI'] = ta.momentum.rsi(df['Close'], window=14)
    df['MACD'] = ta.trend.macd_diff(df['Close'])
    bb_indicator = ta.volatility.BollingerBands(df['Close'])
    df['BB_upper'] = bb_indicator.bollinger_hband()
    df['BB_middle'] = bb_indicator.bollinger_mavg()
    df['BB_lower'] = bb_indicator.bollinger_lband()
    df['ATR'] = ta.volatility.average_true_range(df['High'], df['Low'], df['Close'], window=14)
    df['MFI'] = ta.volume.money_flow_index(df['High'], df['Low'], df['Close'], df['Volume'], window=14)
    
    # Handle NaNs that result from indicator calculations
    df.bfill(inplace=True)
    
    return df

# Directories to save enhanced data
output_dir_daily = '../data/stock_data/enhanced/daily'
output_dir_hourly = '../data/stock_data/enhanced/hourly'
os.makedirs(output_dir_daily, exist_ok=True)
os.makedirs(output_dir_hourly, exist_ok=True)

# Apply outlier handling and feature engineering to all stocks for both daily and hourly data
for stock in stocks:
    try:
        # Process daily data
        daily_data = pd.read_csv(f'../data/stock_data/{stock}_daily.csv', index_col='Date', parse_dates=True)
        
        # Apply outlier handling (IQR-based)
        daily_data = full_feature_engineering_with_outliers_daily(daily_data)
        
        # Apply TA indicators
        daily_data_with_features = apply_ta_features(daily_data)
        
        # Save the final daily data with all features
        daily_data_with_features.to_csv(f'{output_dir_daily}/{stock}_daily_with_features.csv', index=True)
        
        # Process hourly data
        hourly_data = pd.read_csv(f'../data/stock_data/{stock}_hourly.csv', index_col='Datetime', parse_dates=True)
        
        # Apply outlier handling (IQR-based)
        hourly_data = full_feature_engineering_with_outliers_hourly(hourly_data)
        
        # Apply TA indicators
        hourly_data_with_features = apply_ta_features(hourly_data)
        
        # Save the final hourly data with all features
        hourly_data_with_features.to_csv(f'{output_dir_hourly}/{stock}_hourly_with_features.csv', index=True)
        
        print(f"Feature engineering complete for {stock}")
    
    except Exception as e:
        print(f"Error processing {stock}: {e}")

print("Feature engineering complete for all stocks and data saved.")


In [ ]:
# Cell 8(a): Inspect the data after applying technical analysis features for all stocks (Daily and Hourly)

def inspect_ta_engineered_data_for_all_stocks(stocks, data_dir_daily, data_dir_hourly):
    for stock in stocks:
        # Inspect Daily Data
        file_path_daily = f'{data_dir_daily}/{stock}_daily_with_features.csv'
        print(f"\nInspecting Technical Analysis Features for {stock} (Daily Data)")
        
        # Load the enhanced daily data
        data_daily = pd.read_csv(file_path_daily, index_col='Date', parse_dates=True)
        
        # Display the first few rows of the dataframe
        print(f"First few rows of {stock} daily data:")
        print(data_daily.head())
        
        # Check for missing values in daily data
        missing_values_daily = data_daily.isnull().sum()
        if missing_values_daily.sum() > 0:
            print(f"\nMissing values in {stock} daily data:")
            print(missing_values_daily[missing_values_daily > 0])
        else:
            print(f"\nNo missing values found in {stock} daily dataset.")
        
        # Summary statistics for daily data
        print(f"\nSummary statistics of {stock} daily data:")
        print(data_daily.describe())
        
        # Plot closing price and a few selected indicators for daily data
        plt.figure(figsize=(14, 7))
        plt.plot(data_daily['Close'], label='Close Price')
        
        if 'SMA_20' in data_daily.columns:
            plt.plot(data_daily['SMA_20'], label='20-day SMA')
        if 'RSI' in data_daily.columns:
            plt.plot(data_daily['RSI'], label='RSI')
        if 'MACD' in data_daily.columns:
            plt.plot(data_daily['MACD'], label='MACD')
        
        plt.title(f'{stock} Daily Close Price and Selected Indicators')
        plt.legend()
        plt.show()

        # Correlation matrix for daily data
        correlation_matrix_daily = data_daily.corr()
        print(f"\nCorrelation matrix of {stock} daily features:")
        print(correlation_matrix_daily)

        # Visualize the correlation matrix for daily data
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlation_matrix_daily, annot=True, cmap='coolwarm', linewidths=0.5)
        plt.title(f'Correlation Matrix of {stock} Daily Technical Analysis Features')
        plt.show()

        # Inspect Hourly Data
        file_path_hourly = f'{data_dir_hourly}/{stock}_hourly_with_features.csv'
        print(f"\nInspecting Technical Analysis Features for {stock} (Hourly Data)")

        # Load the enhanced hourly data
        data_hourly = pd.read_csv(file_path_hourly, index_col='Datetime', parse_dates=True)

        # Display the first few rows of the hourly dataframe
        print(f"First few rows of {stock} hourly data:")
        print(data_hourly.head())
        
        # Check for missing values in hourly data
        missing_values_hourly = data_hourly.isnull().sum()
        if missing_values_hourly.sum() > 0:
            print(f"\nMissing values in {stock} hourly data:")
            print(missing_values_hourly[missing_values_hourly > 0])
        else:
            print(f"\nNo missing values found in {stock} hourly dataset.")

        # Summary statistics for hourly data
        print(f"\nSummary statistics of {stock} hourly data:")
        print(data_hourly.describe())
        
        # Plot closing price and a few selected indicators for hourly data
        plt.figure(figsize=(14, 7))
        plt.plot(data_hourly['Close'], label='Close Price')
        
        if 'SMA_20' in data_hourly.columns:
            plt.plot(data_hourly['SMA_20'], label='20-hour SMA')
        if 'RSI' in data_hourly.columns:
            plt.plot(data_hourly['RSI'], label='RSI')
        if 'MACD' in data_hourly.columns:
            plt.plot(data_hourly['MACD'], label='MACD')
        
        plt.title(f'{stock} Hourly Close Price and Selected Indicators')
        plt.legend()
        plt.show()
        
        # Correlation matrix for hourly data
        correlation_matrix_hourly = data_hourly.corr()
        print(f"\nCorrelation matrix of {stock} hourly features:")
        print(correlation_matrix_hourly)
        
        # Visualize the correlation matrix for hourly data
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlation_matrix_hourly, annot=True, cmap='coolwarm', linewidths=0.5)
        plt.title(f'Correlation Matrix of {stock} Hourly Technical Analysis Features')
        plt.show()

# Apply to all stocks
data_dir_daily = '../data/stock_data/enhanced/daily'
data_dir_hourly = '../data/stock_data/enhanced/hourly'
inspect_ta_engineered_data_for_all_stocks(stocks, data_dir_daily, data_dir_hourly)


In [ ]:
# Cell 8(b): Inspect the data after applying outlier feature engineering for all stocks (Daily and Hourly)

def inspect_outlier_engineered_data_for_all_stocks(stocks, data_dir_daily, data_dir_hourly):
    for stock in stocks:
        # Inspect Daily Data
        file_path_daily = f'{data_dir_daily}/{stock}_daily_with_features.csv'
        print(f"\nInspecting Outlier Features for {stock} (Daily Data)")
        
        # Load the enhanced daily data
        data_daily = pd.read_csv(file_path_daily, index_col='Date', parse_dates=True)
        
        # Display the first few rows of the dataframe
        print(f"First few rows of {stock} daily data:")
        print(data_daily.head())
        
        # Check for missing values in daily data
        missing_values_daily = data_daily.isnull().sum()
        if missing_values_daily.sum() > 0:
            print(f"\nMissing values in {stock} daily data:")
            print(missing_values_daily[missing_values_daily > 0])
        else:
            print(f"\nNo missing values found in {stock} daily dataset.")
        
        # Summary statistics for daily data
        print(f"\nSummary statistics of {stock} daily data:")
        print(data_daily.describe())
        
        # Plot closing price and outlier-related features for daily data
        plt.figure(figsize=(14, 7))
        plt.plot(data_daily['Close'], label='Close Price')
        
        if 'RollingMean_10' in data_daily.columns:
            plt.plot(data_daily['RollingMean_10'], label='10-day Rolling Mean')
        if 'RollingStd_10' in data_daily.columns:
            plt.plot(data_daily['RollingStd_10'], label='10-day Rolling Std Dev')
        if 'LogReturn' in data_daily.columns:
            plt.plot(data_daily['LogReturn'], label='Log Return')
        if 'DaysSinceLastOutlier' in data_daily.columns:
            plt.plot(data_daily['DaysSinceLastOutlier'], label='Days Since Last Outlier')
        if 'OutlierFreq_30' in data_daily.columns:
            plt.plot(data_daily['OutlierFreq_30'], label='Outlier Frequency (Last 30 Days)')

        plt.title(f'{stock} Daily Close Price and Outlier-Related Features')
        plt.legend()
        plt.show()
        
        # Correlation matrix for daily data
        correlation_matrix_daily = data_daily.corr()
        print(f"\nCorrelation matrix of {stock} daily features:")
        print(correlation_matrix_daily)
        
        # Visualize the correlation matrix for daily data
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlation_matrix_daily, annot=True, cmap='coolwarm', linewidths=0.5)
        plt.title(f'Correlation Matrix of {stock} Daily Outlier Features')
        plt.show()
        
        # Inspect Hourly Data
        file_path_hourly = f'{data_dir_hourly}/{stock}_hourly_with_features.csv'
        print(f"\nInspecting Outlier Features for {stock} (Hourly Data)")
        
        # Load the enhanced hourly data
        data_hourly = pd.read_csv(file_path_hourly, index_col='Datetime', parse_dates=True)
        
        # Display the first few rows of the hourly dataframe
        print(f"First few rows of {stock} hourly data:")
        print(data_hourly.head())
        
        # Check for missing values in hourly data
        missing_values_hourly = data_hourly.isnull().sum()
        if missing_values_hourly.sum() > 0:
            print(f"\nMissing values in {stock} hourly data:")
            print(missing_values_hourly[missing_values_hourly > 0])
        else:
            print(f"\nNo missing values found in {stock} hourly dataset.")
        
        # Summary statistics for hourly data
        print(f"\nSummary statistics of {stock} hourly data:")
        print(data_hourly.describe())
        
        # Plot closing price and outlier-related features for hourly data
        plt.figure(figsize=(14, 7))
        plt.plot(data_hourly['Close'], label='Close Price')
        
        if 'RollingMean_10' in data_hourly.columns:
            plt.plot(data_hourly['RollingMean_10'], label='10-hour Rolling Mean')
        if 'RollingStd_10' in data_hourly.columns:
            plt.plot(data_hourly['RollingStd_10'], label='10-hour Rolling Std Dev')
        if 'LogReturn' in data_hourly.columns:
            plt.plot(data_hourly['LogReturn'], label='Log Return')
        if 'DaysSinceLastOutlier' in data_hourly.columns:
            plt.plot(data_hourly['DaysSinceLastOutlier'], label='Days Since Last Outlier')
        if 'OutlierFreq_30' in data_hourly.columns:
            plt.plot(data_hourly['OutlierFreq_30'], label='Outlier Frequency (Last 30 Hours)')

        plt.title(f'{stock} Hourly Close Price and Outlier-Related Features')
        plt.legend()
        plt.show()

        # Correlation matrix for hourly data
        correlation_matrix_hourly = data_hourly.corr()
        print(f"\nCorrelation matrix of {stock} hourly features:")
        print(correlation_matrix_hourly)

        # Visualize the correlation matrix for hourly data
        plt.figure(figsize=(10, 8))
        sns.heatmap(correlation_matrix_hourly, annot=True, cmap='coolwarm', linewidths=0.5)
        plt.title(f'Correlation Matrix of {stock} Hourly Outlier Features')
        plt.show()

# Apply to all stocks
data_dir_daily = '../data/stock_data/enhanced/daily'
data_dir_hourly = '../data/stock_data/enhanced/hourly'
inspect_outlier_engineered_data_for_all_stocks(stocks, data_dir_daily, data_dir_hourly)

In [ ]:
# Cell 9 : Scaling Data

# Directory to save the scaler files
model_save_dir = "../models"
os.makedirs(model_save_dir, exist_ok=True)

# Initialize dictionaries to store the scaled data
scaled_daily_data_dict = {}
scaled_daily_outlier_features_dict = {}
scaled_daily_ta_features_dict = {}
scaled_hourly_data_dict = {}
scaled_hourly_outlier_features_dict = {}
scaled_hourly_ta_features_dict = {}

def scale_dataframe(df, scaler_name):
    """
    Apply Standard Scaling to the DataFrame, excluding categorical features like 'OutlierFlag' if they exist.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing stock data with columns to be scaled.
    scaler_name (str): Name to save the scaler.
    
    Returns:
    pd.DataFrame: Scaled DataFrame.
    """
    scaler = StandardScaler()
    
    # Select numerical columns for scaling
    features_to_scale = df.select_dtypes(include=[np.number])
    
    # Exclude 'OutlierFlag' if it exists
    if 'OutlierFlag' in df.columns:
        features_to_scale = features_to_scale.drop(columns=['OutlierFlag'], errors='ignore')
    
    # If there are no numerical features left after dropping 'OutlierFlag', return the original DataFrame
    if features_to_scale.empty:
        print(f"No numerical features to scale for {scaler_name}.")
        return df
    
    # Scale the features
    scaled_features = pd.DataFrame(scaler.fit_transform(features_to_scale), columns=features_to_scale.columns, index=df.index)
    
    # Save the scaler in the models directory
    scaler_path = os.path.join(model_save_dir, f'{scaler_name}_scaler.pkl')
    joblib.dump(scaler, scaler_path)
    
    # Combine scaled features with 'OutlierFlag' and other non-numerical columns if they exist
    non_scaled_columns = df.drop(columns=features_to_scale.columns, errors='ignore')
    scaled_df = pd.concat([scaled_features, non_scaled_columns], axis=1)
    
    return scaled_df

# Applying scaling and saving scalers
for stock in stocks:
    # Scale and store the data
    scaled_daily_data_dict[stock] = scale_dataframe(daily_data_dict[stock], f'{stock}_daily')
    scaled_daily_outlier_features_dict[stock] = scale_dataframe(daily_outlier_features_dict[stock], f'{stock}_daily_outlier')
    scaled_daily_ta_features_dict[stock] = scale_dataframe(daily_ta_features_dict[stock], f'{stock}_daily_ta')
    
    scaled_hourly_data_dict[stock] = scale_dataframe(hourly_data_dict[stock], f'{stock}_hourly')
    scaled_hourly_outlier_features_dict[stock] = scale_dataframe(hourly_outlier_features_dict[stock], f'{stock}_hourly_outlier')
    scaled_hourly_ta_features_dict[stock] = scale_dataframe(hourly_ta_features_dict[stock], f'{stock}_hourly_ta')

print("Scaling complete, and data stored in dictionaries. Scalers saved in the models directory.")

In [ ]:
# Print column names for all scaled dataframes

print("Columns in scaled daily data (scaled_daily_data_dict):")
for stock in scaled_daily_data_dict:
    print(f"\n{stock}:")
    print(scaled_daily_data_dict[stock].columns.tolist())

print("\nColumns in scaled daily outlier-featured data (scaled_daily_outlier_features_dict):")
for stock in scaled_daily_outlier_features_dict:
    print(f"\n{stock}:")
    print(scaled_daily_outlier_features_dict[stock].columns.tolist())

print("\nColumns in scaled daily TA-featured data (scaled_daily_ta_features_dict):")
for stock in scaled_daily_ta_features_dict:
    print(f"\n{stock}:")
    print(scaled_daily_ta_features_dict[stock].columns.tolist())

print("\nColumns in scaled hourly data (scaled_hourly_data_dict):")
for stock in scaled_hourly_data_dict:
    print(f"\n{stock}:")
    print(scaled_hourly_data_dict[stock].columns.tolist())

print("\nColumns in scaled hourly outlier-featured data (scaled_hourly_outlier_features_dict):")
for stock in scaled_hourly_outlier_features_dict:
    print(f"\n{stock}:")
    print(scaled_hourly_outlier_features_dict[stock].columns.tolist())

print("\nColumns in scaled hourly TA-featured data (scaled_hourly_ta_features_dict):")
for stock in scaled_hourly_ta_features_dict:
    print(f"\n{stock}:")
    print(scaled_hourly_ta_features_dict[stock].columns.tolist())


In [ ]:
# Cell 10: Train, Test, Split Models

# Function to create sequences for LSTM
def create_sequences(data, sequence_length):
    sequences = []
    targets = []
    for i in range(len(data) - sequence_length):
        sequence = data[i:i+sequence_length, :]
        target = data[i+sequence_length, 0]
        sequences.append(sequence)
        targets.append(target)
    return np.array(sequences), np.array(targets)

# Function to build LSTM model
def build_lstm_model(input_shape):
    model = Sequential()
    model.add(LSTM(units=50, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.2))
    model.add(LSTM(units=50))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

# Sequence lengths
sequence_length_daily = 30  # 30 days
sequence_length_hourly = 24  # 24 hours

# Create directories for saving models
model_save_dir = "../models"
os.makedirs(model_save_dir, exist_ok=True)

# Aggregated data across all stocks for each type of model
# --- Daily Data Models ---
aggregated_daily_orig = []
aggregated_daily_outlier = []
aggregated_daily_ta = []

# --- Hourly Data Models ---
aggregated_hourly_orig = []
aggregated_hourly_outlier = []
aggregated_hourly_ta = []

# Aggregate data across all stocks
for stock in stocks:
    aggregated_daily_orig.extend(scaled_daily_data_dict[stock].values)
    aggregated_daily_outlier.extend(scaled_daily_outlier_features_dict[stock].values)
    aggregated_daily_ta.extend(scaled_daily_ta_features_dict[stock].values)
    
    aggregated_hourly_orig.extend(scaled_hourly_data_dict[stock].values)
    aggregated_hourly_outlier.extend(scaled_hourly_outlier_features_dict[stock].values)
    aggregated_hourly_ta.extend(scaled_hourly_ta_features_dict[stock].values)

# Convert lists to numpy arrays
aggregated_daily_orig = np.array(aggregated_daily_orig)
aggregated_daily_outlier = np.array(aggregated_daily_outlier)
aggregated_daily_ta = np.array(aggregated_daily_ta)
aggregated_hourly_orig = np.array(aggregated_hourly_orig)
aggregated_hourly_outlier = np.array(aggregated_hourly_outlier)
aggregated_hourly_ta = np.array(aggregated_hourly_ta)

# 1. Model for original daily time series data
X_daily_orig, y_daily_orig = create_sequences(aggregated_daily_orig, sequence_length_daily)
X_train_daily_orig, X_test_daily_orig, y_train_daily_orig, y_test_daily_orig = train_test_split(X_daily_orig, y_daily_orig, test_size=0.2, shuffle=False)
model_daily_orig = build_lstm_model((X_train_daily_orig.shape[1], X_train_daily_orig.shape[2]))
model_daily_orig.fit(X_train_daily_orig, y_train_daily_orig, epochs=10, batch_size=32)
loss_daily_orig = model_daily_orig.evaluate(X_test_daily_orig, y_test_daily_orig)
print(f'Model Loss (Original Daily): {loss_daily_orig}')
model_daily_orig.save(os.path.join(model_save_dir, 'model_daily_orig.keras'))

# 2. Model for daily outlier feature-engineered data
X_daily_outlier, y_daily_outlier = create_sequences(aggregated_daily_outlier, sequence_length_daily)
X_train_daily_outlier, X_test_daily_outlier, y_train_daily_outlier, y_test_daily_outlier = train_test_split(X_daily_outlier, y_daily_outlier, test_size=0.2, shuffle=False)
model_daily_outlier = build_lstm_model((X_train_daily_outlier.shape[1], X_train_daily_outlier.shape[2]))
model_daily_outlier.fit(X_train_daily_outlier, y_train_daily_outlier, epochs=10, batch_size=32)
loss_daily_outlier = model_daily_outlier.evaluate(X_test_daily_outlier, y_test_daily_outlier)
print(f'Model Loss (Outlier-Engineered Daily): {loss_daily_outlier}')
model_daily_outlier.save(os.path.join(model_save_dir, 'model_daily_outlier.keras'))

# 3. Model for daily technical analysis feature-engineered data
X_daily_ta, y_daily_ta = create_sequences(aggregated_daily_ta, sequence_length_daily)
X_train_daily_ta, X_test_daily_ta, y_train_daily_ta, y_test_daily_ta = train_test_split(X_daily_ta, y_daily_ta, test_size=0.2, shuffle=False)
model_daily_ta = build_lstm_model((X_train_daily_ta.shape[1], X_train_daily_ta.shape[2]))
model_daily_ta.fit(X_train_daily_ta, y_train_daily_ta, epochs=10, batch_size=32)
loss_daily_ta = model_daily_ta.evaluate(X_test_daily_ta, y_test_daily_ta)
print(f'Model Loss (Technical Analysis Daily): {loss_daily_ta}')
model_daily_ta.save(os.path.join(model_save_dir, 'model_daily_ta.keras'))

# 4. Model for original hourly time series data
X_hourly_orig, y_hourly_orig = create_sequences(aggregated_hourly_orig, sequence_length_hourly)
X_train_hourly_orig, X_test_hourly_orig, y_train_hourly_orig, y_test_hourly_orig = train_test_split(X_hourly_orig, y_hourly_orig, test_size=0.2, shuffle=False)
model_hourly_orig = build_lstm_model((X_train_hourly_orig.shape[1], X_train_hourly_orig.shape[2]))
model_hourly_orig.fit(X_train_hourly_orig, y_train_hourly_orig, epochs=10, batch_size=32)
loss_hourly_orig = model_hourly_orig.evaluate(X_test_hourly_orig, y_test_hourly_orig)
print(f'Model Loss (Original Hourly): {loss_hourly_orig}')
model_hourly_orig.save(os.path.join(model_save_dir, 'model_hourly_orig.keras'))

# 5. Model for hourly outlier feature-engineered data
X_hourly_outlier, y_hourly_outlier = create_sequences(aggregated_hourly_outlier, sequence_length_hourly)
X_train_hourly_outlier, X_test_hourly_outlier, y_train_hourly_outlier, y_test_hourly_outlier = train_test_split(X_hourly_outlier, y_hourly_outlier, test_size=0.2, shuffle=False)
model_hourly_outlier = build_lstm_model((X_train_hourly_outlier.shape[1], X_train_hourly_outlier.shape[2]))
model_hourly_outlier.fit(X_train_hourly_outlier, y_train_hourly_outlier, epochs=10, batch_size=32)
loss_hourly_outlier = model_hourly_outlier.evaluate(X_test_hourly_outlier, y_test_hourly_outlier)
print(f'Model Loss (Outlier-Engineered Hourly): {loss_hourly_outlier}')
model_hourly_outlier.save(os.path.join(model_save_dir, 'model_hourly_outlier.keras'))

# 6. Model for hourly technical analysis feature-engineered data
X_hourly_ta, y_hourly_ta = create_sequences(aggregated_hourly_ta, sequence_length_hourly)
X_train_hourly_ta, X_test_hourly_ta, y_train_hourly_ta, y_test_hourly_ta = train_test_split(X_hourly_ta, y_hourly_ta, test_size=0.2, shuffle=False)
model_hourly_ta = build_lstm_model((X_train_hourly_ta.shape[1], X_train_hourly_ta.shape[2]))
model_hourly_ta.fit(X_train_hourly_ta, y_train_hourly_ta, epochs=10, batch_size=32)
loss_hourly_ta = model_hourly_ta.evaluate(X_test_hourly_ta, y_test_hourly_ta)
print(f'Model Loss (Technical Analysis Hourly): {loss_hourly_ta}')
model_hourly_ta.save(os.path.join(model_save_dir, 'model_hourly_ta.keras'))

print("Training and saving models completed.")

In [ ]:
# Cell 11: Evaluate Models

def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    predictions = np.squeeze(predictions)
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = mean_squared_error(y_test, predictions, squared=False)
    mape = mean_absolute_percentage_error(y_test, predictions)
    direction_correct = (np.sign(predictions[1:] - predictions[:-1]) == np.sign(y_test[1:] - y_test[:-1])).mean()
    return mse, mae, rmse, mape, direction_correct, predictions

def plot_predictions(y_test, predictions, data_type, stock, save_dir=None):
    plt.figure(figsize=(14, 7))
    plt.plot(y_test, label='Actual')
    plt.plot(predictions, label='Predicted')
    plt.title(f'{stock} - Actual vs Predicted ({data_type} Model)')
    plt.xlabel('Time')
    plt.ylabel('Stock Price')
    plt.legend()
    if save_dir:
        plt.savefig(f"{save_dir}/{stock}_{data_type}_predictions.png")
    plt.show()

# Initialize dictionaries to store evaluation results
evaluation_results = {}
model_save_dir = "../models"
plot_save_dir = "../plots"
os.makedirs(plot_save_dir, exist_ok=True)

# Load the models once
models = {
    "Original Daily": (model_daily_orig, X_test_daily_orig, y_test_daily_orig),
    "Outlier Features Daily": (model_daily_outlier, X_test_daily_outlier, y_test_daily_outlier),
    "Technical Analysis Daily": (model_daily_ta, X_test_daily_ta, y_test_daily_ta),
    "Original Hourly": (model_hourly_orig, X_test_hourly_orig, y_test_hourly_orig),
    "Outlier Features Hourly": (model_hourly_outlier, X_test_hourly_outlier, y_test_hourly_outlier),
    "Technical Analysis Hourly": (model_hourly_ta, X_test_hourly_ta, y_test_hourly_ta)
}

# Evaluate models for each stock and data type
for stock in stocks:
    evaluation_results[stock] = {}
    
    for data_type, (model, X_test, y_test) in models.items():
        try:
            mse, mae, rmse, mape, direction_correct, predictions = evaluate_model(model, X_test, y_test)
            evaluation_results[stock][data_type] = {
                'MSE': mse, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'Directional Accuracy': direction_correct
            }
            plot_predictions(y_test, predictions, data_type, stock, save_dir=plot_save_dir)
        except Exception as e:
            print(f"Error evaluating {data_type} for {stock}: {e}")

# Save evaluation results to a JSON file
with open(os.path.join(model_save_dir, 'evaluation_results.json'), 'w') as f:
    json.dump(evaluation_results, f, indent=4)

# Display the evaluation results
for stock, results in evaluation_results.items():
    print(f"\nEvaluation Results for {stock}:")
    for data_type, metrics in results.items():
        print(f"  {data_type}: MSE={metrics['MSE']}, MAE={metrics['MAE']}, RMSE={metrics['RMSE']}, MAPE={metrics['MAPE']}, Directional Accuracy={metrics['Directional Accuracy']}")

print(f"Total Evaluation Time: {time.time() - start_time:.2f} seconds")

In [ ]:
# Cell 12: Further Analysis of Models

# Function to perform extended evaluation
def extended_evaluation(model, X_test, y_test):
    predictions = model.predict(X_test)
    predictions = np.squeeze(predictions)
    
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = mean_squared_error(y_test, predictions, squared=False)
    mape = mean_absolute_percentage_error(y_test, predictions)
    evs = explained_variance_score(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    
    direction_correct = (np.sign(predictions[1:] - predictions[:-1]) == np.sign(y_test[1:] - y_test[:-1])).mean()
    
    return {
        'MSE': mse, 
        'MAE': mae, 
        'RMSE': rmse, 
        'MAPE': mape, 
        'EVS': evs, 
        'R2': r2, 
        'Directional Accuracy': direction_correct,
        'Predictions': predictions
    }

# Extended analysis results
extended_results = {}

# Analyze each model's performance
for model_name, (model, X_test, y_test) in models.items():
    print(f"Evaluating {model_name}...")
    extended_results[model_name] = extended_evaluation(model, X_test, y_test)

# Convert results to a DataFrame for easier analysis
df_results = pd.DataFrame(extended_results).T
df_results.drop(columns='Predictions', inplace=True)  # Drop predictions column for readability

# Display the extended analysis results
print("\nExtended Evaluation Results:\n", df_results)

# Save extended results to a CSV file
df_results.to_csv(os.path.join(model_save_dir, 'extended_evaluation_results.csv'), index=True)

# Correlation Matrix of Evaluation Metrics
plt.figure(figsize=(10, 8))
sns.heatmap(df_results.corr(), annot=True, cmap="coolwarm", linewidths=0.5)
plt.title('Correlation Matrix of Evaluation Metrics')
plt.show()

# Function to plot residuals
def plot_residuals(y_test, predictions, model_name, save_dir=None):
    if len(y_test) != len(predictions):
        print(f"Skipping residuals plot for {model_name} due to length mismatch.")
        return
    residuals = y_test - predictions
    plt.figure(figsize=(10, 6))
    sns.histplot(residuals, kde=True, color='blue', bins=50)
    plt.title(f'Residuals Distribution for {model_name}')
    plt.xlabel('Residuals')
    plt.ylabel('Frequency')
    if save_dir:
        plt.savefig(f"{save_dir}/{model_name}_residuals.png")
    plt.show()

# Plot residuals for each model
for model_name, (model, X_test, y_test) in models.items():
    predictions = extended_results[model_name]['Predictions']
    plot_residuals(y_test, predictions, model_name, save_dir=plot_save_dir)

In [ ]:
# Cell 12: Further Analysis of Models

# Function to perform extended evaluation
def extended_evaluation(model, X_test, y_test):
    predictions = model.predict(X_test)
    predictions = np.squeeze(predictions)
    
    mse = mean_squared_error(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    rmse = mean_squared_error(y_test, predictions, squared=False)
    mape = mean_absolute_percentage_error(y_test, predictions)
    evs = explained_variance_score(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    
    direction_correct = (np.sign(predictions[1:] - predictions[:-1]) == np.sign(y_test[1:] - y_test[:-1])).mean()
    
    return {
        'MSE': mse, 
        'MAE': mae, 
        'RMSE': rmse, 
        'MAPE': mape, 
        'EVS': evs, 
        'R2': r2, 
        'Directional Accuracy': direction_correct,
        'Predictions': predictions
    }

# Extended analysis results
extended_results = {}

# Analyze each model's performance
for model_name, (model, X_test, y_test) in models.items():
    print(f"Evaluating {model_name}...")
    extended_results[model_name] = extended_evaluation(model, X_test, y_test)

# Convert results to a DataFrame for easier analysis
df_results = pd.DataFrame(extended_results).T
df_results.drop(columns='Predictions', inplace=True)  # Drop predictions column for readability

# Ensure all data is numeric
df_results = df_results.apply(pd.to_numeric, errors='coerce')

# Drop any columns or rows that became NaN after coercion
df_results.dropna(axis=1, inplace=True)

# Display the extended analysis results
print("\nExtended Evaluation Results:\n", df_results)

# Save extended results to a CSV file
df_results.to_csv(os.path.join(model_save_dir, 'extended_evaluation_results.csv'), index=True)

# Correlation Matrix of Evaluation Metrics
plt.figure(figsize=(10, 8))
sns.heatmap(df_results.corr(), annot=True, cmap="coolwarm", linewidths=0.5)
plt.title('Correlation Matrix of Evaluation Metrics')
plt.show()

# Function to plot residuals
def plot_residuals(y_test, predictions, model_name, save_dir=None):
    if len(y_test) != len(predictions):
        print(f"Skipping residuals plot for {model_name} due to length mismatch.")
        return
    residuals = y_test - predictions
    plt.figure(figsize=(10, 6))
    sns.histplot(residuals, kde=True, color='blue', bins=50)
    plt.title(f'Residuals Distribution for {model_name}')
    plt.xlabel('Residuals')
    plt.ylabel('Frequency')
    if save_dir:
        plt.savefig(f"{save_dir}/{model_name}_residuals.png")
    plt.show()

# Plot residuals for each model
for model_name, (model, X_test, y_test) in models.items():
    predictions = extended_results[model_name]['Predictions']
    plot_residuals(y_test, predictions, model_name, save_dir=plot_save_dir)

In [ ]:
# Cell 13: Ensemble Models (Separate Daily and Hourly Ensembles)

# Function to create an ensemble prediction by averaging predictions from different models
def ensemble_predictions(models, X_tests):
    ensemble_pred = np.zeros_like(models[0].predict(X_tests[0]))  # Initialize with zeros
    for model, X_test in zip(models, X_tests):
        ensemble_pred += model.predict(X_test)
    ensemble_pred /= len(models)  # Average the predictions
    return np.squeeze(ensemble_pred)

# Separate models and test data for daily and hourly ensembling
models_daily_to_ensemble = [model_daily_orig, model_daily_outlier, model_daily_ta]
X_tests_daily_to_ensemble = [X_test_daily_orig, X_test_daily_outlier, X_test_daily_ta]

models_hourly_to_ensemble = [model_hourly_orig, model_hourly_outlier, model_hourly_ta]
X_tests_hourly_to_ensemble = [X_test_hourly_orig, X_test_hourly_outlier, X_test_hourly_ta]

# Generate ensemble predictions for daily models
ensemble_preds_daily = ensemble_predictions(models_daily_to_ensemble, X_tests_daily_to_ensemble)

# Evaluate the daily ensemble model
mse_daily = mean_squared_error(y_test_daily_orig, ensemble_preds_daily)
mae_daily = mean_absolute_error(y_test_daily_orig, ensemble_preds_daily)
rmse_daily = np.sqrt(mse_daily)
mape_daily = mean_absolute_percentage_error(y_test_daily_orig, ensemble_preds_daily)
direction_correct_daily = (np.sign(ensemble_preds_daily[1:] - ensemble_preds_daily[:-1]) == np.sign(y_test_daily_orig[1:] - y_test_daily_orig[:-1])).mean()

# Print daily ensemble evaluation results
print(f'Daily Ensemble Model Evaluation:')
print(f'MSE: {mse_daily}')
print(f'MAE: {mae_daily}')
print(f'RMSE: {rmse_daily}')
print(f'MAPE: {mape_daily}')
print(f'Directional Accuracy: {direction_correct_daily}')

# Plot daily ensemble residuals
plot_residuals(y_test_daily_orig, ensemble_preds_daily, "Daily Ensemble Model", save_dir=plot_save_dir)

# Generate ensemble predictions for hourly models
ensemble_preds_hourly = ensemble_predictions(models_hourly_to_ensemble, X_tests_hourly_to_ensemble)

# Evaluate the hourly ensemble model
mse_hourly = mean_squared_error(y_test_hourly_orig, ensemble_preds_hourly)
mae_hourly = mean_absolute_error(y_test_hourly_orig, ensemble_preds_hourly)
rmse_hourly = np.sqrt(mse_hourly)
mape_hourly = mean_absolute_percentage_error(y_test_hourly_orig, ensemble_preds_hourly)
direction_correct_hourly = (np.sign(ensemble_preds_hourly[1:] - ensemble_preds_hourly[:-1]) == np.sign(y_test_hourly_orig[1:] - y_test_hourly_orig[:-1])).mean()

# Print hourly ensemble evaluation results
print(f'Hourly Ensemble Model Evaluation:')
print(f'MSE: {mse_hourly}')
print(f'MAE: {mae_hourly}')
print(f'RMSE: {rmse_hourly}')
print(f'MAPE: {mape_hourly}')
print(f'Directional Accuracy: {direction_correct_hourly}')

# Plot hourly ensemble residuals
plot_residuals(y_test_hourly_orig, ensemble_preds_hourly, "Hourly Ensemble Model", save_dir=plot_save_dir)

In [ ]:
# Cell 14 : Downsampling

# Downsample hourly predictions by slicing to match the number of daily samples
def downsample_hourly_to_daily(hourly_preds, daily_length):
    # Calculate the step size needed to downsample
    step_size = len(hourly_preds) // daily_length
    
    if step_size == 0:
        print("Step size is zero, cannot downsample correctly.")
        return np.array([])  # Return empty array to avoid errors
    
    # Use slicing to downsample
    downsampled_preds = hourly_preds[::step_size]
    
    # Ensure the downsampled data has the exact length as daily data
    return downsampled_preds[:daily_length]

# Resample the hourly predictions
resampled_hourly_preds = downsample_hourly_to_daily(ensemble_preds_hourly, len(y_test_daily_orig))

# Ensure the resampled predictions have the correct shape
resampled_hourly_preds = resampled_hourly_preds.reshape(-1, 1)

# Check the shape of X_test_daily_orig and resampled_hourly_preds
print(f"X_test_daily_orig shape: {X_test_daily_orig.shape}")
print(f"Resampled hourly preds shape after reshape: {resampled_hourly_preds.shape}")

# Repeat the resampled hourly predictions to match the time steps in daily data
ensemble_preds_hourly_expanded = np.repeat(resampled_hourly_preds[:, np.newaxis], X_test_daily_orig.shape[1], axis=1)

# Ensure that the repeated array matches the shape required for concatenation
print(f"Ensemble hourly preds expanded shape: {ensemble_preds_hourly_expanded.shape}")

In [ ]:
# Cell 15: Train the Final Model

# Concatenate the expanded hourly predictions with daily features
X_train_with_hourly = np.concatenate((X_test_daily_orig, ensemble_preds_hourly_expanded), axis=2)

# Rebuild the daily model with the new features
model_daily_with_hourly = build_lstm_model((X_train_with_hourly.shape[1], X_train_with_hourly.shape[2]))

# Split the enriched dataset into training and testing sets
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(X_train_with_hourly, y_test_daily_orig, test_size=0.2, shuffle=False)

# Train the final model
model_daily_with_hourly.fit(X_train_final, y_train_final, epochs=10, batch_size=32)

# Evaluate the final model
final_loss = model_daily_with_hourly.evaluate(X_test_final, y_test_final)
print(f'Final Model Loss: {final_loss}')

# Save the final model
model_daily_with_hourly.save(os.path.join(model_save_dir, 'final_model_daily_with_hourly.keras'))

In [ ]:
# Cell 16: Evaluate the Final Model

# Generate predictions using the final model
final_predictions = model_daily_with_hourly.predict(X_test_final)
final_predictions = np.squeeze(final_predictions)

# Calculate evaluation metrics
final_mse = mean_squared_error(y_test_final, final_predictions)
final_mae = mean_absolute_error(y_test_final, final_predictions)
final_rmse = np.sqrt(final_mse)
final_mape = mean_absolute_percentage_error(y_test_final, final_predictions)
final_direction_correct = (np.sign(final_predictions[1:] - final_predictions[:-1]) == np.sign(y_test_final[1:] - y_test_final[:-1])).mean()

# Print final model evaluation results
print(f'Final Model Evaluation:')
print(f'MSE: {final_mse}')
print(f'MAE: {final_mae}')
print(f'RMSE: {final_rmse}')
print(f'MAPE: {final_mape}')
print(f'Directional Accuracy: {final_direction_correct}')

# Plot final model residuals
plot_residuals(y_test_final, final_predictions, "Final Model with Hourly Data", save_dir=plot_save_dir)

In [ ]:
# Cell 17: Make Predictions and Analyze

# Load the saved scaler (ensure you're using the correct one, e.g., for the daily data)
scaler = joblib.load('../models/AAPL_daily_scaler.pkl')  # Adjust the file path as needed

# Use the final model to make predictions for the upcoming period
future_predictions = model_daily_with_hourly.predict(X_test_final[-1:])

# Reshape and inverse transform the predictions to get them back to the original scale
future_predictions_reshaped = future_predictions.reshape(-1, 1)
future_predictions_inverse = scaler.inverse_transform(np.column_stack([future_predictions_reshaped] * scaler.scale_.shape[0]))[:, 0]

# Inverse transform the actual values for comparison
y_test_final_inverse = scaler.inverse_transform(np.column_stack([y_test_final.reshape(-1, 1)] * scaler.scale_.shape[0]))[:, 0]

# Analyze and interpret the predictions
print(f'Future Predictions (Inverse Transformed): {future_predictions_inverse}')

# You can also visualize the predictions against actual data if available
plt.figure(figsize=(14, 7))

# Plot the last 100 actual values
plt.plot(y_test_final_inverse[-100:], label='Actual')

# Plot the last 100 predicted values (from the final model)
final_predictions = model_daily_with_hourly.predict(X_test_final[-100:])
final_predictions_reshaped = final_predictions.reshape(-1, 1)
final_predictions_inverse = scaler.inverse_transform(np.column_stack([final_predictions_reshaped] * scaler.scale_.shape[0]))[:, 0]
plt.plot(final_predictions_inverse, label='Predicted')

# Plot the future predictions (one step ahead)
plt.plot(np.arange(len(final_predictions_inverse), len(final_predictions_inverse) + len(future_predictions_inverse)),
         future_predictions_inverse, label='Future Prediction', linestyle='--')

plt.title('Final Model Predictions vs Actual')
plt.xlabel('Time')
plt.ylabel('Stock Price')
plt.legend()
plt.show()